# Compute large embeddings

Computes Text-Fabric features for the four local encoders too large to run outside Colab: bge-multilingual-gemma2, Qwen3-Embedding-8B, KaLM-Embedding-Gemma3-12B, Llama-Embed-Nemotron-8B.

Runtime > Change runtime type > pick a GPU runtime with as much VRAM as possible.

| Model | Load dtype | Approx. memory |
| --- | --- | --- |
| bge-multilingual-gemma2 | float16 | ~18.5 GB |
| Qwen3-Embedding-8B | auto (bf16, native) | ~15.1 GB |
| KaLM-Embedding-Gemma3-12B | bfloat16 | ~23.5 GB |
| Llama-Embed-Nemotron-8B | bfloat16 | ~16 GB |

A 40GB A100 fits any one of these plus activations. A 16GB T4 fits only Qwen3-Embedding-8B or Llama-Embed-Nemotron-8B, and even those are tight.

Llama-Embed-Nemotron-8B is licensed for non-commercial, research use only (NVIDIA's customized-nscl-v1).

In [ ]:
!rm -rf /content/tehillim-embeddings
!git clone https://github.com/rdtaylorjr/tehillim-embeddings.git /content/tehillim-embeddings
!cd /content/tehillim-embeddings/programs && pip install .

Set `MODEL_CHOICE` to `"bge"`, `"qwen3"`, `"kalm"`, or `"llama-nemotron"` to run just one model, or leave it `None` to run all four. `generate_local` treats an already-written `.tf` file as done, so this is safe to re-run after a partial failure.

In [ ]:
import subprocess
from pathlib import Path

from semantic.corpus import DEFAULT_BHSA_TF_PATH, Corpus
from semantic.generate import generate_local
from semantic.large_models import ensure_corpus_data, gpu_memory_summary, models_for_choice

MODEL_CHOICE = None


def _clone(url: str, destination: Path) -> None:
    subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)


bhsa_path = ensure_corpus_data(
    bhsa_default=DEFAULT_BHSA_TF_PATH,
    data_dir=Path("/content/_bhsa_data"),
    clone=_clone,
)

corpus = Corpus.load(bhsa_path)
psalms = corpus.psalms()
print(f"{len(psalms)} psalms loaded")

output_root = Path("/content/tehillim-embeddings")
for slug, model_name, torch_dtype in models_for_choice(MODEL_CHOICE):
    print(f"computing {model_name} (torch_dtype={torch_dtype})...")
    written = generate_local(psalms, output_root, slug, torch_dtype=torch_dtype)
    print(f"  wrote {written}")
    summary = gpu_memory_summary()
    if summary:
        print(f"  [GPU memory] {summary}")

print("done")

Download the finished `.tf` files: Colab's file browser, or zip and download in one shot with the next cell. Unzip locally into `tehillim-embeddings/tf/1.0/` and commit.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/tf", "zip", "/content/tehillim-embeddings/tf/1.0")
files.download("/content/tf.zip")